# Gold — Distribuição de clientes por estado e cidade

Desenvolvido por: Ygor Moraes

Este notebook cria a Gold de distribuição geográfica de clientes por estado e cidade.

Fonte:
- Silver `ecommerce_enderecos`

Regra:
- considerar apenas endereços principais;
- contar clientes distintos por estado e cidade.

Granularidade:
- 1 linha por estado e cidade.

Destino ADLS:
- `gold/ecommerce_enderecos/distribuicao_estado_cidade`

Destino SQL Server:
- `squad3.gold_ecommerce_enderecos_distribuicao_estado_cidade`

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# Define imports, caminhos e parâmetros da Gold.

from pyspark.sql.functions import (
    col,
    count,
    countDistinct,
    current_timestamp,
    round as spark_round,
    desc,
    coalesce,
    lit,
    trim,
    upper,
    sum as spark_sum
)

SILVER_ENDERECOS_TABLE = "ecommerce_enderecos"

SILVER_ENDERECOS_PATH = f"{SILVER_BASE_PATH}{SILVER_ENDERECOS_TABLE}"

GOLD_DOMAIN = "ecommerce_enderecos"
GOLD_KPI = "distribuicao_estado_cidade"

GOLD_PATH = f"{GOLD_BASE_PATH}{GOLD_DOMAIN}/{GOLD_KPI}"

FINAL_TABLE_NAME = f"gold_{GOLD_DOMAIN}_{GOLD_KPI}"
FINAL_TABLE = f"{TARGET_SCHEMA}.{FINAL_TABLE_NAME}"

GOLD_WRITE_MODE = "overwrite"

ENDERECOS_REQUIRED_COLUMNS = [
    "id_endereco",
    "id_cliente",
    "estado",
    "cidade",
    "cep",
    "is_principal"
]

GOLD_KEY_COLUMNS = [
    "estado",
    "cidade"
]

adls_options = get_adls_options()

print("Parâmetros definidos com sucesso.")
print("SILVER_ENDERECOS_PATH:", SILVER_ENDERECOS_PATH)
print("GOLD_PATH:", GOLD_PATH)
print("FINAL_TABLE:", FINAL_TABLE)

In [0]:
# Lê a Silver de endereços e valida colunas principais.

df_enderecos = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(SILVER_ENDERECOS_PATH)
)

validate_required_columns(df_enderecos, ENDERECOS_REQUIRED_COLUMNS)

total_enderecos = df_enderecos.count()

duplicados_id_endereco = (
    df_enderecos
    .groupBy("id_endereco")
    .count()
    .filter(col("count") > 1)
    .count()
)

total_clientes = (
    df_enderecos
    .select("id_cliente")
    .distinct()
    .count()
)

total_enderecos_principais = (
    df_enderecos
    .filter(col("is_principal") == True)
    .count()
)

clientes_com_endereco_principal = (
    df_enderecos
    .filter(col("is_principal") == True)
    .select("id_cliente")
    .distinct()
    .count()
)

ceps_nulos = df_enderecos.filter(col("cep").isNull()).count()

print("Silver de endereços lida com sucesso.")
print(f"Total de endereços: {total_enderecos}")
print(f"IDs de endereço duplicados: {duplicados_id_endereco}")
print(f"Total de clientes distintos: {total_clientes}")
print(f"Total de endereços principais: {total_enderecos_principais}")
print(f"Clientes com endereço principal: {clientes_com_endereco_principal}")
print(f"CEPs nulos mantidos: {ceps_nulos}")

df_enderecos.printSchema()

display(
    df_enderecos
    .groupBy("estado")
    .agg(count("*").alias("qtd_enderecos"))
    .orderBy(desc("qtd_enderecos"))
)

In [0]:
# Filtra endereços principais e prepara campos geográficos.

df_enderecos_principais = (
    df_enderecos
    .filter(col("is_principal") == True)
    .select(
        "id_cliente",
        "estado",
        "cidade"
    )
    .withColumn("estado", coalesce(upper(trim(col("estado"))), lit("NAO_INFORMADO")))
    .withColumn("cidade", coalesce(trim(col("cidade")), lit("NAO_INFORMADO")))
)

print("Base de endereços principais preparada.")
print(f"Total de registros: {df_enderecos_principais.count()}")
print(f"Clientes distintos: {df_enderecos_principais.select('id_cliente').distinct().count()}")

display(df_enderecos_principais.limit(10))

In [0]:
# Agrega clientes por estado e cidade.

total_clientes_principais = (
    df_enderecos_principais
    .select("id_cliente")
    .distinct()
    .count()
)

df_gold = (
    df_enderecos_principais
    .groupBy("estado", "cidade")
    .agg(
        countDistinct("id_cliente").alias("qtd_clientes")
    )
    .withColumn(
        "pct_clientes",
        spark_round((col("qtd_clientes") / lit(total_clientes_principais)) * 100, 2)
    )
    .withColumn("gold_processed_at", current_timestamp())
    .orderBy(desc("qtd_clientes"))
)

print("Gold agregada com sucesso.")
print(f"Total de linhas Gold: {df_gold.count()}")

display(df_gold)

In [0]:
# Valida chave da Gold e consistência da contagem de clientes.

total_linhas_gold = df_gold.count()

chaves_duplicadas = (
    df_gold
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_clientes_gold = (
    df_gold
    .agg(spark_sum("qtd_clientes").alias("total_clientes"))
    .collect()[0]["total_clientes"]
)

print(f"Total de linhas Gold: {total_linhas_gold}")
print(f"Chaves duplicadas na Gold: {chaves_duplicadas}")
print(f"Total clientes principais: {total_clientes_principais}")
print(f"Total clientes Gold: {total_clientes_gold}")

if chaves_duplicadas != 0:
    raise ValueError("Validação falhou: existem chaves duplicadas na Gold.")

if total_clientes_gold != total_clientes_principais:
    raise ValueError("Validação falhou: total de clientes da Gold diferente da base principal.")

print("Validação OK: Gold sem duplicidade e com total de clientes consistente.")

In [0]:
# Grava a Gold em Delta no ADLS.

(
    df_gold
    .write
    .format("delta")
    .options(**adls_options)
    .option("overwriteSchema", "true")
    .mode(GOLD_WRITE_MODE)
    .save(GOLD_PATH)
)

print("Gold gravada com sucesso no ADLS.")
print("Caminho:", GOLD_PATH)

In [0]:
# Lê a Gold gravada no ADLS.

df_gold_gravada = (
    spark.read
    .format("delta")
    .options(**adls_options)
    .load(GOLD_PATH)
)

print("Gold lida com sucesso do ADLS.")
print(f"Total de linhas gravadas: {df_gold_gravada.count()}")

display(df_gold_gravada.orderBy(desc("qtd_clientes")))

In [0]:
# Valida a Gold gravada no ADLS.

total_linhas_gold_gravada = df_gold_gravada.count()

chaves_duplicadas_gravada = (
    df_gold_gravada
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_clientes_gold_gravada = (
    df_gold_gravada
    .agg(spark_sum("qtd_clientes").alias("total_clientes"))
    .collect()[0]["total_clientes"]
)

print(f"Total de linhas Gold gravada: {total_linhas_gold_gravada}")
print(f"Chaves duplicadas Gold gravada: {chaves_duplicadas_gravada}")
print(f"Total clientes principais: {total_clientes_principais}")
print(f"Total clientes Gold gravada: {total_clientes_gold_gravada}")

if chaves_duplicadas_gravada != 0:
    raise ValueError("Validação falhou: existem chaves duplicadas na Gold gravada.")

if total_clientes_gold_gravada != total_clientes_principais:
    raise ValueError("Validação falhou: total de clientes da Gold gravada diferente da base principal.")

print("Validação OK: Gold gravada corretamente no ADLS.")

In [0]:
# Prepara a Gold para gravação no SQL Server.

df_gold_sql = (
    df_gold_gravada
    .select(
        "estado",
        "cidade",
        "qtd_clientes",
        "pct_clientes",
        "gold_processed_at"
    )
)

print("Gold preparada para SQL Server.")
print(f"Total de linhas: {df_gold_sql.count()}")
print("Tabela destino:", FINAL_TABLE)

display(df_gold_sql.orderBy(desc("qtd_clientes")))

In [0]:
# Grava a Gold diretamente no SQL Server.

write_sql_table(
    df=df_gold_sql,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    mode="overwrite",
    sql_port=SQL_PORT
)

print("Gold gravada com sucesso no SQL Server.")
print("Tabela:", FINAL_TABLE)

In [0]:
# Lê a tabela final no SQL Server para validação.

df_final_sql = read_sql_table(
    spark=spark,
    sql_host=SQL_HOST,
    sql_database=SQL_DATABASE,
    sql_username=SQL_USERNAME,
    sql_password=SQL_PASSWORD,
    table_name=FINAL_TABLE,
    sql_port=SQL_PORT
)

print("Tabela final lida com sucesso do SQL Server.")
print(f"Total de linhas SQL Server: {df_final_sql.count()}")

display(df_final_sql.orderBy(desc("qtd_clientes")))

In [0]:
# Valida a tabela final no SQL Server.

total_linhas_sql = df_final_sql.count()

chaves_duplicadas_sql = (
    df_final_sql
    .groupBy(GOLD_KEY_COLUMNS)
    .count()
    .filter(col("count") > 1)
    .count()
)

total_clientes_sql = (
    df_final_sql
    .agg(spark_sum("qtd_clientes").alias("total_clientes"))
    .collect()[0]["total_clientes"]
)

print(f"Total de linhas Gold ADLS: {total_linhas_gold_gravada}")
print(f"Total de linhas SQL Server: {total_linhas_sql}")
print(f"Chaves duplicadas SQL Server: {chaves_duplicadas_sql}")
print(f"Total clientes Gold ADLS: {total_clientes_gold_gravada}")
print(f"Total clientes SQL Server: {total_clientes_sql}")

if total_linhas_sql != total_linhas_gold_gravada:
    raise ValueError("Validação falhou: total de linhas no SQL Server diferente da Gold no ADLS.")

if chaves_duplicadas_sql != 0:
    raise ValueError("Validação falhou: existem chaves duplicadas no SQL Server.")

if total_clientes_sql != total_clientes_gold_gravada:
    raise ValueError("Validação falhou: total de clientes no SQL Server diferente da Gold no ADLS.")

print("Validação OK: tabela final SQL Server gravada corretamente.")